# Municipal Ward Hierarchy, Inheritance, and Polymorphic Dispatch

**Session:** 2026-11-03 · OOP Practice Lab  
**Module:** 01 — Python Fundamentals & Algorithmic Thinking

---

## Overview

In this notebook, we implement **Exercise 2**: a complete municipal directory system managing citizens residing in a municipal ward.
We implement:
1. Base class `Person` encapsulating shared demographic attributes (`name`, `yob`).
2. Derived specialized subclasses:
   - `Student`: adds `grade: str`.
   - `Teacher`: adds `subject: str`.
   - `Doctor`: adds `specialist: str`.
3. Polymorphic method overriding of `.describe()`.
4. Municipal container class `Ward` aggregating citizen records, implementing:
   - `add_person(person: Person)`.
   - `describe()`: formats ward header and delegates to polymorphic citizen descriptions.
   - `count_doctor() -> int`: queries the census for medical practitioners.
   - `sort_age()`: sorts residents by ascending age (descending Year of Birth).
   - `ave_teacher_year_of_birth() -> float`: calculates mean birth year of educators.
5. Ingestion and verification against `../data/ward_residents.json`.


## 1. Class Hierarchy Design

We design a clear `is-a` inheritance hierarchy rooted at `Person`. All subclasses invoke `super().__init__(name, yob)` to ensure clean initialization.


In [ ]:
class Person:
    """Base demographic entity representing an individual citizen."""

    def __init__(self, name: str, yob: int) -> None:
        self._name: str = str(name)
        self._yob: int = int(yob)

    def get_name(self) -> str:
        """Return the citizen's full name."""
        return self._name

    def get_yob(self) -> int:
        """Return the Gregorian year of birth."""
        return self._yob

    def describe(self) -> str:
        """Return standard descriptive representation."""
        msg = f"Person - Name: {self._name} - YoB: {self._yob}"
        print(msg)
        return msg


class Student(Person):
    """Specialized citizen representing a student pursuing academic studies."""

    def __init__(self, name: str, yob: int, grade: str) -> None:
        super().__init__(name, yob)
        self.__grade: str = str(grade)

    def get_grade(self) -> str:
        """Return the student's academic grade level."""
        return self.__grade

    def describe(self) -> str:
        """Polymorphic override describing student credentials."""
        msg = f"Student - Name: {self._name} - YoB: {self._yob} - Grade: {self.__grade}"
        print(msg)
        return msg


class Teacher(Person):
    """Specialized citizen representing a professional educator."""

    def __init__(self, name: str, yob: int, subject: str) -> None:
        super().__init__(name, yob)
        self.__subject: str = str(subject)

    def get_subject(self) -> str:
        """Return teaching discipline."""
        return self.__subject

    def describe(self) -> str:
        """Polymorphic override describing teacher credentials."""
        msg = f"Teacher - Name: {self._name} - YoB: {self._yob} - Subject: {self.__subject}"
        print(msg)
        return msg


class Doctor(Person):
    """Specialized citizen representing a medical practitioner."""

    def __init__(self, name: str, yob: int, specialist: str) -> None:
        super().__init__(name, yob)
        self.__specialist: str = str(specialist)

    def get_specialist(self) -> str:
        """Return medical specialization field."""
        return self.__specialist

    def describe(self) -> str:
        """Polymorphic override describing doctor credentials."""
        msg = f"Doctor - Name: {self._name} - YoB: {self._yob} - Specialist: {self.__specialist}"
        print(msg)
        return msg


# Quick verification of polymorphic dispatch
people: list[Person] = [
    Student("Alex", 2005, "12th Grade"),
    Teacher("Dr. Elena", 1978, "Advanced Calculus"),
    Doctor("Dr. Aris", 1980, "Cardiology"),
]

for p in people:
    desc = p.describe()
    assert p.get_name() in desc
print("Polymorphic dispatch verified successfully.")

## 2. Municipal Container Class: `Ward`

The `Ward` class encapsulates an administrative boundary and aggregates a list of `Person` instances.

### Sorting by Age Invariant
Sorting by ascending age ($A_i \le A_j$) is mathematically equivalent to sorting by descending year of birth ($\text{YoB}_i \ge \text{YoB}_j$):
$$A_i = Y_{\text{current}} - \text{YoB}_i \implies A_i < A_j \iff \text{YoB}_i > \text{YoB}_j$$


In [ ]:
class Ward:
    """Administrative municipal ward managing a directory of resident citizens."""

    def __init__(self, name: str) -> None:
        self.__name: str = str(name)
        self.__list_people: list[Person] = []

    def get_name(self) -> str:
        """Return ward municipality designation."""
        return self.__name

    def get_people(self) -> list[Person]:
        """Return a defensive shallow copy of the resident directory."""
        return list(self.__list_people)

    def add_person(self, person: Person) -> None:
        """Enroll a citizen resident into the municipal directory."""
        if not isinstance(person, Person):
            raise TypeError(f"Expected Person instance, received {type(person).__name__}")
        self.__list_people.append(person)

    def describe(self) -> str:
        """Print and return the comprehensive municipal census log."""
        header = f"Ward Name: {self.__name}"
        print(header)
        lines = [header]
        for person in self.__list_people:
            lines.append(person.describe())
        return "\n".join(lines)

    def count_doctor(self) -> int:
        """Count the total number of medical doctors registered in the ward."""
        count = sum(1 for p in self.__list_people if isinstance(p, Doctor))
        return count

    def sort_age(self) -> None:
        """Sort residents in ascending order of age (descending order of YoB)."""
        self.__list_people.sort(key=lambda p: p.get_yob(), reverse=True)

    def ave_teacher_year_of_birth(self) -> float:
        """Compute the mean year of birth among all registered teachers."""
        teacher_yobs = [p.get_yob() for p in self.__list_people if isinstance(p, Teacher)]
        if not teacher_yobs:
            raise ValueError(f"No teachers registered in ward '{self.__name}' to compute average.")
        return sum(teacher_yobs) / len(teacher_yobs)


# Verification with the lecture slide test case
ward = Ward("Ward-01")
ward.add_person(Student("Alice", 2004, "Grade 11"))
ward.add_person(Teacher("Bob", 1980, "Mathematics"))
ward.add_person(Teacher("Charlie", 1970, "Physics"))
ward.add_person(Doctor("David", 1975, "Endocrinology"))
ward.add_person(Doctor("Emma", 1985, "Neurology"))

assert ward.count_doctor() == 2
avg_yob = ward.ave_teacher_year_of_birth()
assert avg_yob == (1980 + 1970) / 2 == 1975.0

# Verify age sorting: youngest first -> highest YoB first
ward.sort_age()
sorted_yobs = [p.get_yob() for p in ward.get_people()]
assert sorted_yobs == [2004, 1985, 1980, 1975, 1970]
print(
    "Lecture test case verified: count_doctor=2, avg_teacher_yob=1975.0, sorted YoB=", sorted_yobs
)

## 3. Real Census Ingestion & Automated Verification

We load the municipal census records stored in `../data/ward_residents.json` to validate full pipeline integration.


In [ ]:
import json
from pathlib import Path

data_path = Path("../data/ward_residents.json")
assert data_path.is_file(), f"Census dataset not found: {data_path}"

with data_path.open("r", encoding="utf-8") as f:
    census_records = json.load(f)

district_ward = Ward("Central Metro District")

for record in census_records:
    role = record["role"].lower()
    name = record["name"]
    yob = record["yob"]
    if role == "student":
        district_ward.add_person(Student(name, yob, record["grade"]))
    elif role == "teacher":
        district_ward.add_person(Teacher(name, yob, record["subject"]))
    elif role == "doctor":
        district_ward.add_person(Doctor(name, yob, record["specialist"]))
    else:
        raise ValueError(f"Unknown census role: {role}")

print(f"Enrolled {len(district_ward.get_people())} residents into {district_ward.get_name()}.")
assert len(district_ward.get_people()) == 9
assert district_ward.count_doctor() == 3

# Compute average teacher year of birth for the district
district_avg_teacher_yob = district_ward.ave_teacher_year_of_birth()
expected_avg = (1978 + 1985 + 1972) / 3.0
assert abs(district_avg_teacher_yob - expected_avg) < 1e-6
print(f"District Teacher Mean YoB: {district_avg_teacher_yob:.2f}")

# Verify age ordering
district_ward.sort_age()
census_yobs = [p.get_yob() for p in district_ward.get_people()]
assert census_yobs == sorted(census_yobs, reverse=True)
print("Census age sorting verified:", census_yobs)